# Stage 2 - Gemma 4 Scenario Generation

This notebook prepares the Colab runtime, selects a local extracted input directory or zip, and runs Gemma 4 inference to produce both a Playwright JSON file and a human-readable scenario summary.

In [ ]:
#@title 1. Prepare repository on the Colab runtime
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/Yongjooon/QA-AI-FineTunning.git"
REPO_NAME = "QA-AI-FineTunning"
REPO_DIR = Path(f"/content/{REPO_NAME}")
OVERRIDE_REPO_DIR = os.environ.get("QA_FINETUNE_REPO_DIR", "").strip()

if OVERRIDE_REPO_DIR and Path(OVERRIDE_REPO_DIR, "pyproject.toml").exists():
    REPO_DIR = Path(OVERRIDE_REPO_DIR)
    print(f"Using override repository directory: {REPO_DIR}")
elif (REPO_DIR / ".git").exists():
    print(f"Repository already exists. Updating: {REPO_DIR}")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"], check=True)
else:
    if REPO_DIR.exists():
        print(f"Directory exists without git metadata. Removing: {REPO_DIR}")
        subprocess.run(["rm", "-rf", str(REPO_DIR)], check=True)
    print(f"Cloning latest repository from {REPO_URL}")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
repo_src = str(REPO_DIR / "src")
os.environ["PYTHONPATH"] = repo_src + os.pathsep + os.environ.get("PYTHONPATH", "")
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)

print("Current working directory:", os.getcwd())
print("PYTHONPATH:", os.environ["PYTHONPATH"])


In [ ]:
#@title 2. Install dependencies
%cd /content/QA-AI-FineTunning
%pip install -q -U pip
%pip uninstall -y transformers
%pip install -q -r requirements-colab.txt
%pip install -q -e . --no-deps

import qafinetune
import transformers
print("qafinetune import ok:", qafinetune.__file__)
print("transformers version:", transformers.__version__)


In [ ]:
#@title 3. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 4. Inference parameters
MODEL_NAME = "google/gemma-4-E2B-it"  # Must match the base model used in training
PERSIST_ROOT = "/content/drive/MyDrive/qa_ai_finetuning"
TRAIN_RUN_NAME = "gemma4_qa_lora_run_01"
ADAPTER_PATH = f"{PERSIST_ROOT}/train_runs/{TRAIN_RUN_NAME}/final_model"
GENERATION_RUN_NAME = "gemma4_qa_generation_01"
MAX_NEW_TOKENS = 700
MAX_INPUT_TOKENS = 1536
TEMPERATURE = 0.0
TOP_P = 1.0

In [ ]:
#@title 5. Inspect Colab runtime
import json
import sys
from pathlib import Path

repo_src = Path('/content/QA-AI-FineTunning/src')
if str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))

from qafinetune.runtime import detect_runtime

runtime_profile = detect_runtime()
print(json.dumps(runtime_profile, indent=2, ensure_ascii=False))

In [ ]:
#@title 6. Select inference input source
from pathlib import Path

INPUT_DATA_SOURCE = "local_dir"  #@param ["local_dir", "drive_path", "upload_widget", "gdown_file_id"]
LOCAL_INPUT_DIR = "/content/QA-AI-FineTunning/data/test_inputs/2026-04-17T09-38-00-6751880b"  #@param {type:"string"}
DRIVE_INPUT_ZIP_PATH = "/content/drive/MyDrive/input_data.zip"  #@param {type:"string"}
GDRIVE_FILE_ID = ""  #@param {type:"string"}
DOWNLOADED_INPUT_ZIP_PATH = "/content/QA-AI-FineTunning/input_data.zip"

if INPUT_DATA_SOURCE == "local_dir":
    INPUT_SOURCE_PATH = LOCAL_INPUT_DIR
elif INPUT_DATA_SOURCE == "drive_path":
    INPUT_SOURCE_PATH = DRIVE_INPUT_ZIP_PATH
elif INPUT_DATA_SOURCE == "upload_widget":
    from google.colab import files
    uploaded = files.upload()
    INPUT_SOURCE_PATH = next(iter(uploaded.keys()))
elif INPUT_DATA_SOURCE == "gdown_file_id":
    if not GDRIVE_FILE_ID.strip():
        raise ValueError("GDRIVE_FILE_ID is empty.")
    import subprocess
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    subprocess.run(["gdown", GDRIVE_FILE_ID.strip(), "-O", DOWNLOADED_INPUT_ZIP_PATH], check=True)
    INPUT_SOURCE_PATH = DOWNLOADED_INPUT_ZIP_PATH
else:
    raise ValueError(f"Unsupported INPUT_DATA_SOURCE: {INPUT_DATA_SOURCE}")

input_source = Path(INPUT_SOURCE_PATH)
if not input_source.exists():
    raise FileNotFoundError(f"Input source was not found: {INPUT_SOURCE_PATH}")

print("Using input source:", input_source)
if input_source.is_dir():
    !find "$INPUT_SOURCE_PATH" -maxdepth 2 -type f | sort | head -n 40
else:
    !ls -lh "$INPUT_SOURCE_PATH"


In [ ]:
#@title 7. Run generation
import os
import shlex
import subprocess
import sys
from pathlib import Path

REPO_DIR = "/content/QA-AI-FineTunning"
input_source_path = Path(INPUT_SOURCE_PATH)
if not input_source_path.exists():
    raise FileNotFoundError(f"Input source was not found: {input_source_path}")

env = os.environ.copy()
env["PYTHONPATH"] = f"{REPO_DIR}/src:" + env.get("PYTHONPATH", "")

command = [
    sys.executable, "-m", "qafinetune.infer",
    "--model_name", MODEL_NAME,
    "--adapter_path", ADAPTER_PATH,
    "--input_source", str(input_source_path),
    "--output_root", PERSIST_ROOT,
    "--run_name", GENERATION_RUN_NAME,
    "--max_new_tokens", str(MAX_NEW_TOKENS),
    "--max_input_tokens", str(MAX_INPUT_TOKENS),
    "--temperature", str(TEMPERATURE),
    "--top_p", str(TOP_P),
]

print(" ".join(shlex.quote(x) for x in command))
print("PYTHONPATH =", env["PYTHONPATH"])
print("INPUT_SOURCE_PATH =", input_source_path)

process = subprocess.Popen(
    command,
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
)

assert process.stdout is not None
for line in process.stdout:
    print(line, end="")

return_code = process.wait()
print("\nRETURN CODE:", return_code)
if return_code != 0:
    raise RuntimeError(f"Generation failed with exit code {return_code}")


In [ ]:
#@title 8. Preview generated outputs
from pathlib import Path

run_dir = Path(PERSIST_ROOT) / "generated_runs" / GENERATION_RUN_NAME
summary_path = run_dir / "scenario_summary.md"
json_path = run_dir / "playwright_scenario.json"

if not summary_path.exists() or not json_path.exists():
    raise FileNotFoundError(f"Expected output files were not found under: {run_dir}")

print("Summary path:", summary_path)
print(summary_path.read_text(encoding='utf-8'))
print("\nJSON path:", json_path)
print(json_path.read_text(encoding='utf-8')[:4000])